# ARC NeuroGolf static ONNX solver

Reference layout adapted from the uploaded fill/additive-marking notebook. The task-specific modelling cell uses a semantic feature-tree or a symbolic reflection builder, not raw output-template lookup.

In [1]:
!rm -rf /kaggle/working/*
%reset -f

In [2]:
COMPETITION = '/kaggle/input/competitions/neurogolf-2026'

In [3]:
import importlib.util, subprocess, sys
missing=[p for p in ['onnx','onnxruntime','onnxscript','torch','numpy'] if importlib.util.find_spec(p) is None]
if missing:
    subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])
print('dependencies ok')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 50.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 722.0/722.0 kB 30.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 7.6 MB/s eta 0:00:00
dependencies ok


In [4]:
import json, os, time, hashlib, zipfile,  csv, base64
import glob, sys,math, random, collections,io,shutil
from pathlib import Path
import numpy as np
import torch
import onnx
import onnxruntime as ort
import torch, torch.nn as nn, torch.nn.functional as F
from collections import defaultdict, Counter
from onnx import shape_inference,helper

In [5]:
torch.set_num_threads(1)
TASK_ID='task085'
CH=10
H=W=30
LOCAL_TASK=Path('/mnt/data/task085.json')
KAGGLE_TASK=Path(COMPETITION)/f'{TASK_ID}.json'
TASK_PATH=LOCAL_TASK if LOCAL_TASK.exists() else KAGGLE_TASK
WORK=Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('/mnt/data')
OUT_DIR=WORK/'task085_v12_generator_exact_onnx'
OUT_DIR.mkdir(parents=True,exist_ok=True)
ONNX_PATH=OUT_DIR/f'{TASK_ID}.onnx'
SUBMISSION_PATH=WORK/'submission.zip'
SUMMARY_PATH=OUT_DIR/'task085_v12_validation_summary.json'
task=json.loads(TASK_PATH.read_text())
print(TASK_PATH,{k:len(task.get(k,[])) for k in ['train','test','arc-gen']})

/kaggle/input/competitions/neurogolf-2026/task085.json {'train': 2, 'test': 1, 'arc-gen': 262}


In [6]:
def grid_to_tensor(grid, offset=(0,0)):
    a=np.asarray(grid,dtype=np.int64)
    h,w=a.shape
    x=np.zeros((1,CH,H,W),dtype=np.float32)
    ro,co=offset
    rr,cc=np.indices((h,w))
    rr=rr+ro; cc=cc+co
    valid=(rr>=0)&(rr<H)&(cc>=0)&(cc<W)
    x[0,a[valid],rr[valid],cc[valid]]=1.0
    return x

def tensor_exact(y, expected):
    return np.array_equal((y>0.5).astype(np.float32), expected)

def tensor_to_grid(y,h,w):
    return y[0,:,:h,:w].argmax(axis=0).astype(int).tolist()

In [7]:
class Task085V12(nn.Module):
    def __init__(self):
        super().__init__()
        self.register_buffer('rr',torch.arange(30,dtype=torch.float32).view(1,1,30,1))
        self.register_buffer('cc',torch.arange(30,dtype=torch.float32).view(1,1,1,30))
        self.register_buffer('color_idx',torch.arange(10,dtype=torch.float32).view(1,10,1,1))

    @staticmethod
    def near(a,b):
        return (torch.abs(a-b)<0.25).float()

    def forward(self,x):
        active=x.sum(dim=1,keepdim=True).clamp(0,1)
        m=x

        row_has=m.amax(dim=3,keepdim=True)
        col_has=m.amax(dim=2,keepdim=True)
        rmin=torch.where(row_has>0.5,self.rr,torch.full_like(self.rr,30.0)).amin(dim=2,keepdim=True)
        rmax=torch.where(row_has>0.5,self.rr,torch.full_like(self.rr,-1.0)).amax(dim=2,keepdim=True)
        cmin=torch.where(col_has>0.5,self.cc,torch.full_like(self.cc,30.0)).amin(dim=3,keepdim=True)
        cmax=torch.where(col_has>0.5,self.cc,torch.full_like(self.cc,-1.0)).amax(dim=3,keepdim=True)

        hh=rmax-rmin+1.0
        ww=cmax-cmin+1.0
        count=m.sum(dim=(2,3),keepdim=True)
        nonempty=(count>0.5).float()
        solid=self.near(count,hh*ww)
        odd_h=(torch.remainder(hh,2.0)>0.5).float()
        odd_w=(torch.remainder(ww,2.0)>0.5).float()

        horizontal=nonempty*solid*self.near(hh,3.0)*odd_w*(ww>2.5).float()
        vertical=nonempty*solid*self.near(ww,3.0)*odd_h*(hh>2.5).float()
        valid_object=(horizontal+vertical).clamp(0,1)

        # The background is the largest active color whose support is not one valid bar.
        # This remains correct even when bar pixels outnumber background pixels.
        bg_candidate=nonempty*(1-valid_object)
        candidate_count=count*bg_candidate
        max_candidate_count=candidate_count.amax(dim=1,keepdim=True)
        bg_mask=bg_candidate*self.near(count,max_candidate_count)
        # deterministic low-index tie break for malformed/non-generator inputs
        masked_idx=torch.where(bg_mask>0.5,self.color_idx,torch.full_like(self.color_idx,10.0))
        min_idx=masked_idx.amin(dim=1,keepdim=True)
        bg_mask=bg_mask*self.near(self.color_idx,min_idx)

        horizontal=horizontal*(1-bg_mask)
        vertical=vertical*(1-bg_mask)

        rel_col=self.cc-cmin
        rel_row=self.rr-rmin
        odd_col=(torch.remainder(rel_col,2.0)>0.5).float()
        odd_row=(torch.remainder(rel_row,2.0)>0.5).float()
        middle_row=self.near(self.rr,rmin+1.0)
        middle_col=self.near(self.cc,cmin+1.0)

        carve_h=m*horizontal*middle_row*odd_col
        carve_v=m*vertical*middle_col*odd_row
        carve=(carve_h+carve_v).amax(dim=1,keepdim=True).clamp(0,1)

        y=x*(1-carve)+bg_mask*carve
        return y*active

model=Task085V12().eval()
print(model)

Task085V12()


In [8]:
def eval_examples(examples, runner):
    oks=[]
    for ex in examples:
        x=grid_to_tensor(ex['input'])
        expected=grid_to_tensor(ex['output'])
        y=runner(x)
        oks.append(tensor_exact(y,expected))
    return sum(oks),len(oks)

with torch.no_grad():
    torch_runner=lambda x:model(torch.from_numpy(x)).numpy()
    for split in ['train','test','arc-gen']:
        print('torch',split,eval_examples(task[split],torch_runner))

torch train (2, 2)
torch test (1, 1)
torch arc-gen (262, 262)


In [9]:
# Independent generator-aligned stress cases.
def make_case(h,w,bg,bars):
    gi=np.full((h,w),bg,dtype=int)
    go=gi.copy()
    for orient,color,r,c,length in bars:
        if orient=='h':
            gi[r:r+3,c:c+length]=color
            go[r:r+3,c:c+length]=color
            go[r+1,c+1:c+length:2]=bg
        else:
            gi[r:r+length,c:c+3]=color
            go[r:r+length,c:c+3]=color
            go[r+1:r+length:2,c+1]=bg
    return {'input':gi.tolist(),'output':go.tolist()}

stress=[]
# Every ordered background/object color pair and both orientations.
for bg in range(10):
    for color in range(10):
        if color==bg: continue
        stress.append(make_case(13,17,bg,[('h',color,1,1,15)]))
        stress.append(make_case(17,13,bg,[('v',color,1,1,15)]))
# Every odd length from 3 through 29, including edge-touching bars.
for length in range(3,30,2):
    stress.append(make_case(30,30,9,[('h',0,0,30-length,length)]))
    stress.append(make_case(30,30,8,[('v',1,30-length,0,length)]))
# Dense foreground-majority configurations with all nine non-background colors.
for bg in range(10):
    colors=[c for c in range(10) if c!=bg]
    bars=[('h',color,3*i,0,29) for i,color in enumerate(colors)]
    stress.append(make_case(30,30,bg,bars))
# Mixed horizontal/vertical cases, kept non-overlapping.
rng=random.Random(85012)
for _ in range(200):
    h=rng.randint(10,30); w=rng.randint(10,30); bg=rng.randrange(10)
    colors=[c for c in range(10) if c!=bg]; rng.shuffle(colors)
    gi=np.full((h,w),bg,dtype=int); bars=[]
    occupied=np.zeros((h,w),bool)
    for color in colors[:rng.randint(1,7)]:
        for _trial in range(100):
            orient=rng.choice(['h','v'])
            if orient=='h':
                possible=[v for v in range(3,w+1,2)]
                if not possible: continue
                length=rng.choice(possible); r=rng.randint(0,h-3); c=rng.randint(0,w-length)
                sl=(slice(r,r+3),slice(c,c+length))
            else:
                possible=[v for v in range(3,h+1,2)]
                if not possible: continue
                length=rng.choice(possible); r=rng.randint(0,h-length); c=rng.randint(0,w-3)
                sl=(slice(r,r+length),slice(c,c+3))
            if not occupied[sl].any():
                occupied[sl]=True; bars.append((orient,color,r,c,length)); break
    stress.append(make_case(h,w,bg,bars))

with torch.no_grad():
    print('torch stress',eval_examples(stress,torch_runner))
print('stress cases',len(stress))

torch stress (418, 418)
stress cases 418


In [10]:
dummy=torch.zeros((1,CH,H,W),dtype=torch.float32)
dummy[:,0,:,:]=1.0
torch.onnx.export(
    model,dummy,ONNX_PATH,
    input_names=['input'],output_names=['output'],
    opset_version=17,do_constant_folding=True,
    dynamic_axes=None
)
onnx_model=onnx.load(str(ONNX_PATH))
onnx.checker.check_model(onnx_model)
ops=collections.Counter(n.op_type for n in onnx_model.graph.node)
forbidden=sorted(set(ops)&{'Loop','Scan','NonZero','Unique','Script','Function'})
print('onnx bytes',ONNX_PATH.stat().st_size,'nodes',sum(ops.values()),'forbidden',forbidden)
print('ops',ops)

W0710 13:06:14.431000 15 torch/onnx/_internal/exporter/_compat.py:125] Setting ONNX exporter to use operator set version 18 because the requested opset_version 17 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


[torch.onnx] Obtain model graph for `Task085V12()` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `Task085V12()` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...


/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
The model version conversion is not supported by the onnxscript version converter and fallback is enabled. The model will be converted using the onnx C API (target version: 17).


[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
onnx bytes 12025 nodes 117 forbidden []
ops Counter({'Mul': 27, 'Sub': 18, 'Cast': 14, 'Greater': 10, 'Add': 7, 'Abs': 7, 'Less': 7, 'ReduceMax': 6, 'Where': 5, 'Div': 4, 'Floor': 4, 'Clip': 3, 'ReduceMin': 3, 'ReduceSum': 2})


In [11]:
so=ort.SessionOptions()
so.intra_op_num_threads=1
so.inter_op_num_threads=1
sess=ort.InferenceSession(str(ONNX_PATH),sess_options=so,providers=['CPUExecutionProvider'])
iname=sess.get_inputs()[0].name
ort_runner=lambda x:sess.run(None,{iname:x})[0]
results={}
for split in ['train','test','arc-gen']:
    results[split]=eval_examples(task[split],ort_runner)
    print('ort',split,results[split])
results['stress']=eval_examples(stress,ort_runner)
print('ort stress',results['stress'])
# shifted active canvases
shifted=[]
for ex in task['train']+task['test']:
    h=len(ex['input']); w=len(ex['input'][0])
    for off in [(1,0),(0,1),(2,2)]:
        if h+off[0]<=30 and w+off[1]<=30:
            x=grid_to_tensor(ex['input'],off)
            e=grid_to_tensor(ex['output'],off)
            shifted.append(tensor_exact(ort_runner(x),e))
print('shifted',sum(shifted),len(shifted))
# runtime
x0=grid_to_tensor(task['test'][0]['input'])
for _ in range(5): ort_runner(x0)
t0=time.perf_counter()
for _ in range(100): ort_runner(x0)
print('ORT 100 runs seconds',time.perf_counter()-t0)

ort train (2, 2)
ort test (1, 1)
ort arc-gen (262, 262)
ort stress (418, 418)
shifted 7 7
ORT 100 runs seconds 0.015895010999997794


In [12]:
assert results['train']==(len(task['train']),len(task['train']))
assert results['test']==(len(task['test']),len(task['test']))
assert results['arc-gen']==(len(task['arc-gen']),len(task['arc-gen']))
assert results['stress']==(len(stress),len(stress))
assert all(shifted)
assert ONNX_PATH.stat().st_size < 1_400_000
assert not forbidden
with zipfile.ZipFile(SUBMISSION_PATH,'w',compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(ONNX_PATH,arcname=f'{TASK_ID}.onnx')
summary={
    'task_id':TASK_ID,
    'model':'v12_generator_exact_arbitrary_background_orientation',
    'source_arc_task':'3bdb4ada',
    'results':{k:list(v) for k,v in results.items()},
    'shifted':[sum(shifted),len(shifted)],
    'stress_cases':len(stress),
    'onnx_bytes':ONNX_PATH.stat().st_size,
    'onnx_nodes':sum(ops.values()),
    'forbidden_ops':forbidden,
    'submission':str(SUBMISSION_PATH),
}
SUMMARY_PATH.write_text(json.dumps(summary,indent=2))
print(json.dumps(summary,indent=2))
print('wrote',SUBMISSION_PATH)

{
  "task_id": "task085",
  "model": "v12_generator_exact_arbitrary_background_orientation",
  "source_arc_task": "3bdb4ada",
  "results": {
    "train": [
      2,
      2
    ],
    "test": [
      1,
      1
    ],
    "arc-gen": [
      262,
      262
    ],
    "stress": [
      418,
      418
    ]
  },
  "shifted": [
    7,
    7
  ],
  "stress_cases": 418,
  "onnx_bytes": 12025,
  "onnx_nodes": 117,
  "forbidden_ops": [],
  "submission": "/kaggle/working/submission.zip"
}
wrote /kaggle/working/submission.zip
